# 펌프 · 헤드 — 실무자 조치 vs 모델 경고 검토

설계·규칙은 [`../CM_검토_설계.md`](../CM_검토_설계.md). 이 노트북은 그 흐름을 펌프 · 헤드에 적용한다.

- 경고 대상: `kind=pump` (P1–P7, I1–I7) + 헤드 인젝션 릴레이 (`HEAD_*`, `injection_time`)
- 펌프 경고 로그는 10-15 18:58 부터, 헤드 릴레이는 10-19 06:50 부터.
- 설비키는 `PUMP_P7|HEAD_HD4` 처럼 헤드를 같이 붙인다. `match_on='pump'` 은 같은 펌프, `'head'` 는 같은 헤드의 짝 펌프다.
  헤드 저항 경고는 POL·ISO 두 펌프에 같이 뜨므로 `head` 짝도 의미가 있다.

**순서**: 1 입력 → 2 짝짓기 후보 → 3 원시 태그 → 4 재생 피처 → 5 라벨 기록 → 6 집계
판정은 5번 라벨 CSV에 적는다. 노트북 출력은 근거일 뿐이다.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from review_lib import paths, match, db, plot

DOMAIN = "pump"
CHECK = paths.CHECK_DIRS[DOMAIN]
FIGS = CHECK / "figs"; FIGS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)

## 1. 입력

먼저 한 번: `cd model_review_1103 && ../venv/bin/python warning/build_inputs.py`
(일지 xlsx 나 `action_map.csv` 를 고쳤으면 다시 돌린다)

In [ ]:
acts = pd.read_csv(paths.ACTIONS_DIR / f"actions_{DOMAIN}.csv", parse_dates=["start_dt", "end_dt"])
inc = match.incidents(acts)

ep_all = pd.read_csv(paths.EPISODES_DIR / "warning_episodes.csv",
                     parse_dates=["onset", "last_detected", "cleared", "first_logged"])
ep = ep_all[(ep_all["domain"] == DOMAIN) & (ep_all["last_detected"] >= paths.REVIEW_START)].reset_index(drop=True)
print(f"사건 {len(inc)}개 · 경고 에피소드 {len(ep)}건")
inc[["incident_id", "작업일", "start_dt", "device_key", "sub_unit", "wagon_no", "model_scope", "설비상세", "조치", "작업IDs"]]

In [ ]:
# 대상·사유별 경고 건수 — 어떤 설비가 얼마나 자주 울리는지
ep.groupby(["key", "reason_first"]).agg(건수=("episode_id", "size"), 총시간_h=("duration_h", "sum")).round(1)

In [ ]:
ax = plot.timeline(ep, inc, row_col="key", start=paths.REVIEW_START, end=paths.REVIEW_END)
ax.figure.savefig(FIGS / "timeline.png", dpi=120, bbox_inches="tight")

## 2. 짝짓기 후보

시간창(초안): 조치 시작 전 **7일** 안에 떠 있던 경고 = `phase=before` (사전 탐지 후보),
조치 후 **24시간** 안에 처음 뜬 경고 = `phase=after` (사후 — 재가동 과도 또는 조치 무효).
`lead_h` = 경고 onset → 조치 시작 (시간). 후보일 뿐이다. 사유·부위가 맞는지는 사람이 본다.

In [ ]:
cand = match.candidates(inc, ep)
print(cand.groupby(["incident_id", "phase"]).size().unstack(fill_value=0))

In [ ]:
cand[cand["match_on"] == "pump"]

In [ ]:
# 짝 후보가 하나도 없는 사건 → FN 후보 (또는 PRE/OUT)
no_cand = inc[~inc["incident_id"].isin(cand.loc[cand["phase"] == "before", "incident_id"])]
no_cand[["incident_id", "start_dt", "device_key", "model_scope", "설비상세"]]

In [ ]:
# 어떤 사건과도 짝이 안 된 경고 → FP / FIELD_CHECK 후보
ep_orphan = ep[~ep["episode_id"].isin(cand["episode_id"])]
print(f"짝 없는 경고 {len(ep_orphan)} / {len(ep)}")
ep_orphan.groupby(["key", "reason_first"]).size().sort_values(ascending=False).head(30)

## 3. 사건별 확인 — 원시 태그 (DB)

조치 시점에 우리 모델 입력 태그가 실제로 움직였는지 본다. 하루 단위로 캐시되므로 두 번째부터는 DB에 가지 않는다.
**현장 DB 다** — 넓은 구간을 한 번에 뽑지 않는다.

In [ ]:
TAGS = db.model_tags("pump")
INC_ID = "P01"                                  # ← 볼 사건
r = inc.set_index("incident_id").loc[INC_ID]
t0, t1 = pd.Timestamp(r["start_dt"]), r["end_dt"]
raw = db.fetch_range(t0 - pd.Timedelta(days=2), t0 + pd.Timedelta(days=1), TAGS, group="pump")
raw.shape

In [ ]:
pumps = [k.split("_")[1] for k in r["device_key"].split("|") if k.startswith("PUMP_")]
show = [t for t in TAGS if any(t.endswith("_" + p) for p in pumps)] + ["Auto_Running_PL"]
onsets = cand.loc[cand["incident_id"] == INC_ID, "onset"]
fig = plot.around(raw, show, t0, t1, onsets, title=f"{INC_ID} {r['설비상세'][:40]}", pad=pd.Timedelta(hours=12))
fig.savefig(FIGS / f"{INC_ID}_raw.png", dpi=110, bbox_inches="tight")

### 4. 재생 — 모델이 본 피처

운영 펌프 모델에 원시 데이터를 다시 흘려 사이클별 피처(`zn_*`)와 점수를 얻는다.
피처 8개: `pump_resid` `r_phys` `t_self` `decel_half_sec` `release_half_sec` `ft_cv` `pt_std` `P0`.
창 판정이 비어 있지 않게, 보고 싶은 시각보다 반나절 이상 앞에서 시작한다 (위 `raw` 는 조치 2일 전부터).

In [ ]:
from review_lib import replay

cyc = replay.pump_cycles(raw, pumps=pumps)
daily = replay.pump_daily(cyc)
daily

## 5. 라벨 기록

`labels_incidents.csv` (사건 한 행) · `labels_episodes.csv` (경고 한 행) 를 엑셀로 열어 채운다.
이 셀은 새 사건·경고 행만 추가하고 **이미 적은 라벨은 건드리지 않는다**. 값 정의는 설계 문서 4절.

In [ ]:
inc_info = ["domain", "작업일", "start_dt", "device_key", "model_scope", "expected_reasons", "설비상세", "조치", "작업IDs"]
lab_inc = match.sync_labels(inc, CHECK / "labels_incidents.csv", "incident_id", match.INCIDENT_LABEL_COLS, inc_info)

ep_info = ["kind", "key", "device_key", "sub_unit", "onset", "last_detected", "duration_h", "reasons", "max_level", "max_sev", "message_first"]
lab_ep = match.sync_labels(ep, CHECK / "labels_episodes.csv", "episode_id", match.EPISODE_LABEL_COLS, ep_info)
lab_inc.head()

## 6. 집계

In [ ]:
lab_inc = pd.read_csv(CHECK / "labels_incidents.csv", dtype=str, keep_default_na=False)
lab_ep = pd.read_csv(CHECK / "labels_episodes.csv", dtype=str, keep_default_na=False)
match.confusion_table(lab_inc, lab_ep)